# Week 4: Multilingual retrieval over official UK guidance (Colab version)

Self-contained: run **Runtime > Run all**. It installs two libraries, writes the project code, collects the 38 official pages live from GOV.UK and NHS.uk, builds the 150-question test set and reruns the week 4 experiment. A CPU runtime is enough (about 3-5 minutes; a GPU runtime is faster).

Because the pages are fetched live, numbers can differ slightly from the 28 Oct 2026 run if a page has changed since.

In [ ]:
%pip install -q sentence-transformers faiss-cpu rank-bm25

## Project code
The same files as the project's `src/` and `eval/` folders.

In [ ]:
import os, sys
for d in ['src', 'eval', 'data/raw', 'data/processed', 'results']:
    os.makedirs(d, exist_ok=True)

In [ ]:
%%writefile src/sources.py
"""Curated list of official sources for the three domains.

Every entry was checked against the live site on 28 Oct 2026 (HTTP 200).
GOV.UK pages are fetched through the public Content API; NHS pages as HTML.
Both are published under the Open Government Licence v3.0, and NHS reuse
also needs attribution, a link back and a date notice (nhs.uk terms).
"""

GOVUK = {
    "schools": [
        "schools-admissions",
        "apply-for-primary-school-place",
        "apply-for-secondary-school-place",
        "school-attendance-absence",
        "apply-free-school-meals",
        "children-with-special-educational-needs",
        "free-school-transport",
        "school-uniform",
        "types-of-school",
        "school-term-holiday-dates",
        "know-when-you-can-leave-school",
        "complain-about-school",
    ],
    "housing": [
        "private-renting",
        "private-renting-evictions",
        "guidance/renters-rights-act-overview-for-tenants",
        "tenancy-deposit-protection",
        "council-housing",
        "housing-association-homes",
        "council-housing-association-evictions",
        "housing-benefit",
        "universal-credit",
        "homelessness-help-from-council",
        "council-tax",
        "apply-council-tax-reduction",
    ],
    "health": [
        "help-nhs-costs",
        "healthcare-immigration-application",
    ],
}

NHS = {
    "health": [
        "https://www.nhs.uk/nhs-services/gps/how-to-register-with-a-gp-surgery/",
        "https://www.nhs.uk/nhs-services/gps/gp-appointments-and-bookings/",
        "https://www.nhs.uk/nhs-services/urgent-and-emergency-care-services/when-to-use-111/",
        "https://www.nhs.uk/nhs-services/urgent-and-emergency-care-services/when-to-go-to-ae/",
        "https://www.nhs.uk/nhs-services/urgent-and-emergency-care-services/when-to-call-999/",
        "https://www.nhs.uk/nhs-services/dentists/how-to-find-an-nhs-dentist/",
        "https://www.nhs.uk/nhs-services/dentists/how-much-nhs-dental-treatment-costs/",
        "https://www.nhs.uk/nhs-services/pharmacies/how-pharmacies-can-help/",
        "https://www.nhs.uk/nhs-services/help-with-health-costs/nhs-low-income-scheme-lis/",
        "https://www.nhs.uk/nhs-services/visiting-or-moving-to-england/how-to-access-nhs-services-in-england-if-you-are-visiting-from-abroad/",
        "https://www.nhs.uk/vaccinations/nhs-vaccinations-and-when-to-have-them/",
        "https://www.nhs.uk/pregnancy/finding-out/what-to-do-when-you-find-out-you-are-pregnant/",
    ],
}


In [ ]:
%%writefile src/ingest.py
"""Fetch every source page, keep a raw snapshot, and write one clean record per page.

Output: data/raw/<id>.(json|html) and data/processed/pages.jsonl
Each record keeps the section structure (h2/h3 headings) so chunks can carry
their heading, plus three dates: the publisher's own date, the publisher's
last technical update, and when we fetched it (for data-currency reporting).
"""
import hashlib
import json
import pathlib
import time
from datetime import datetime, timezone

import requests
from bs4 import BeautifulSoup

from sources import GOVUK, NHS

ROOT = pathlib.Path(__file__).resolve().parents[1]
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed" / "pages.jsonl"
UA = {"User-Agent": "MSc research prototype (multilingual civic assistant); polite crawler"}

# GOV.UK formats store text in different fields; order follows the rendered page.
GOVUK_TEXT_FIELDS = ["introduction", "introductory_paragraph", "body", "more_information",
                     "need_to_know", "other_ways_to_apply", "what_you_need_to_know"]


def page_id(url):
    return hashlib.sha1(url.encode()).hexdigest()[:10]


def sections_from_html(html, page_title):
    """Split HTML into (heading, text) sections at h2/h3 boundaries."""
    soup = BeautifulSoup(html, "html.parser")
    sections, heading, buf = [], page_title, []
    for el in soup.find_all(["h2", "h3", "p", "li", "td", "th"]):
        if el.name in ("h2", "h3"):
            if buf:
                sections.append({"heading": heading, "text": " ".join(buf)})
            heading, buf = el.get_text(" ", strip=True), []
        elif not el.find_parent(["li", "td", "th"]) or el.name in ("td", "th"):
            # skip nested elements already covered by their parent's text
            t = el.get_text(" ", strip=True)
            if t:
                buf.append(t)
    if buf:
        sections.append({"heading": heading, "text": " ".join(buf)})
    return sections


def fetch_govuk(path, domain):
    r = requests.get("https://www.gov.uk/api/content/" + path, headers=UA, timeout=30)
    r.raise_for_status()
    data = r.json()
    url = "https://www.gov.uk/" + path
    (RAW / f"{page_id(url)}.json").write_text(r.text)
    det = data.get("details", {})
    html = "".join(det.get(f, "") for f in GOVUK_TEXT_FIELDS if isinstance(det.get(f), str))
    for part in det.get("parts", []):
        html += f"<h2>{part['title']}</h2>{part['body']}"
    title = data.get("title", "")
    return {
        "id": page_id(url), "url": url, "source": "GOV.UK", "domain": domain,
        "title": title, "format": data.get("schema_name"),
        "public_updated_at": (data.get("public_updated_at") or "")[:10],
        "updated_at": (data.get("updated_at") or "")[:10],
        "sections": sections_from_html(html, title),
    }


def fetch_nhs(url, domain):
    r = requests.get(url, headers=UA, timeout=30)
    r.raise_for_status()
    (RAW / f"{page_id(url)}.html").write_text(r.text)
    soup = BeautifulSoup(r.text, "html.parser")
    reviewed = soup.find(string=lambda s: s and "Page last reviewed" in s)
    reviewed = reviewed.strip().split(":", 1)[1].strip() if reviewed else ""
    modified = ""
    for s in soup.find_all("script", type="application/ld+json"):
        try:
            modified = (json.loads(s.string or "{}").get("dateModified") or modified)[:10]
        except (ValueError, AttributeError):
            pass
    article = soup.find("article") or soup.find("main")
    for junk in article.find_all(["aside", "nav", "footer", "script", "style"]):
        junk.decompose()
    title = soup.find("h1").get_text(" ", strip=True)
    # the lead sentence ("999 is for life-threatening emergencies...") sits outside <article>
    lede = soup.find(class_="nhsuk-lede-text")
    html = (str(lede) if lede and article not in lede.parents else "") + str(article)
    return {
        "id": page_id(url), "url": url, "source": "NHS.uk", "domain": domain,
        "title": title, "format": "nhs_article",
        "public_updated_at": datetime.strptime(reviewed, "%d %B %Y").strftime("%Y-%m-%d") if reviewed else "",
        "updated_at": modified,
        "sections": sections_from_html(html, title),
    }


def main():
    RAW.mkdir(parents=True, exist_ok=True)
    OUT.parent.mkdir(parents=True, exist_ok=True)
    fetched_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    jobs = [(fetch_govuk, p, d) for d, ps in GOVUK.items() for p in ps]
    jobs += [(fetch_nhs, u, d) for d, us in NHS.items() for u in us]
    records, failed = [], []
    for fn, target, domain in jobs:
        try:
            rec = fn(target, domain)
            rec["fetched_at"] = fetched_at
            rec["words"] = sum(len(s["text"].split()) for s in rec["sections"])
            records.append(rec)
            print(f"ok   {rec['domain']:8} {rec['words']:5}w  {rec['title']}")
        except Exception as e:  # keep going; report at the end
            failed.append((target, repr(e)))
            print(f"FAIL {target}: {e}")
        time.sleep(0.5)
    with OUT.open("w") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"\n{len(records)} pages written to {OUT.relative_to(ROOT)}; {len(failed)} failed")
    for t, e in failed:
        print("  failed:", t, e)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/chunk.py
"""Turn pages into retrievable chunks that respect the page's own sections.

Rules (kept simple so they are easy to describe in the methodology):
  * one chunk per h2/h3 section;
  * sections shorter than MIN_WORDS are merged into the next one on the same page;
  * sections longer than MAX_WORDS are split into windows with OVERLAP words;
  * every chunk text starts with "<page title> - <section heading>:" so the
    embedding sees the context a reader would see.
Output: data/processed/chunks.csv
"""
import json
import pathlib

import pandas as pd

ROOT = pathlib.Path(__file__).resolve().parents[1]
MIN_WORDS, MAX_WORDS, OVERLAP = 40, 200, 40


def merge_small(sections):
    merged, carry = [], None
    for s in sections:
        if carry:
            s = {"heading": carry["heading"] + " / " + s["heading"], "text": carry["text"] + " " + s["text"]}
            carry = None
        if len(s["text"].split()) < MIN_WORDS:
            carry = s
        else:
            merged.append(s)
    if carry:  # trailing short section joins the previous one, or stands alone
        if merged:
            merged[-1] = {"heading": merged[-1]["heading"] + " / " + carry["heading"],
                          "text": merged[-1]["text"] + " " + carry["text"]}
        else:
            merged.append(carry)
    return merged


def windows(words):
    if len(words) <= MAX_WORDS:
        return [words]
    step = MAX_WORDS - OVERLAP
    return [words[i:i + MAX_WORDS] for i in range(0, len(words) - OVERLAP, step)]


def main():
    pages = [json.loads(l) for l in open(ROOT / "data/processed/pages.jsonl")]
    rows = []
    for p in pages:
        for s_i, s in enumerate(merge_small(p["sections"])):
            for w_i, w in enumerate(windows(s["text"].split())):
                body = " ".join(w)
                rows.append({
                    "chunk_id": f"{p['id']}-{s_i:02d}-{w_i}",
                    "page_id": p["id"], "url": p["url"], "source": p["source"], "domain": p["domain"],
                    "title": p["title"], "heading": s["heading"],
                    "public_updated_at": p["public_updated_at"], "fetched_at": p["fetched_at"],
                    "words": len(w), "body": body,
                    "text": f"{p['title']} - {s['heading']}: {body}",
                })
    df = pd.DataFrame(rows)
    df.to_csv(ROOT / "data/processed/chunks.csv", index=False)
    print(f"{len(pages)} pages -> {len(df)} chunks")
    print(df.groupby("domain").agg(pages=("page_id", "nunique"), chunks=("chunk_id", "count"),
                                   words=("words", "sum")).to_string())
    print("chunk words: median", int(df.words.median()), "min", df.words.min(), "max", df.words.max())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/retrieval.py
"""Dense and lexical retrievers over the chunk table.

Dense: sentence-transformers model + FAISS inner-product index on normalised
vectors (= cosine similarity). E5 models need "query: " / "passage: " prefixes.
Lexical: BM25 (rank_bm25), used as a non-neural baseline.
"""
import re

import faiss
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer


class DenseRetriever:
    def __init__(self, model_name, texts):
        self.model = SentenceTransformer(model_name)
        self.e5 = "e5" in model_name.lower()
        passages = [("passage: " + t) if self.e5 else t for t in texts]
        emb = self.model.encode(passages, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
        self.index = faiss.IndexFlatIP(emb.shape[1])
        self.index.add(np.asarray(emb, dtype="float32"))

    def search(self, query, k=10):
        q = self.model.encode([("query: " + query) if self.e5 else query], normalize_embeddings=True)
        scores, ids = self.index.search(np.asarray(q, dtype="float32"), k)
        return list(zip(ids[0].tolist(), scores[0].tolist()))

    def save(self, path):
        faiss.write_index(self.index, str(path))


def tokenize(text):
    return re.findall(r"\w+", text.lower())


class BM25Retriever:
    def __init__(self, texts):
        self.bm25 = BM25Okapi([tokenize(t) for t in texts])

    def search(self, query, k=10):
        scores = self.bm25.get_scores(tokenize(query))
        ids = np.argsort(-scores)[:k]
        return [(int(i), float(scores[i])) for i in ids]


In [ ]:
%%writefile eval/build_queries.py
"""Build the retrieval test set and check every answer phrase against the corpus.

Each English question has:
  gold_url - the page that answers it
  evidence - a short phrase copied verbatim from that page that contains the answer
A retrieved chunk only counts as "answer-bearing" if it contains the evidence
phrase, so a hit on the right page but the wrong section is not counted.

Translations (pl, ro, ar, ur) were written by the author and are flagged
"unverified" until a native speaker checks them.
Output: eval/queries.csv
"""
import csv
import json
import pathlib
import sys

ROOT = pathlib.Path(__file__).resolve().parents[1]
G = "https://www.gov.uk/"
N = "https://www.nhs.uk/nhs-services/"

# qid: (domain, gold_url, evidence, {lang: question})
Q = {
    "S01": ("schools", G + "schools-admissions", "close on 15 January", {
        "en": "When is the deadline to apply for a primary school place?",
        "pl": "Jaki jest termin składania wniosku o miejsce w szkole podstawowej?",
        "ro": "Care este termenul limită pentru a depune cererea pentru un loc la școala primară?",
        "ar": "ما هو الموعد النهائي للتقديم على مقعد في المدرسة الابتدائية؟",
        "ur": "پرائمری اسکول میں داخلے کی درخواست دینے کی آخری تاریخ کیا ہے؟"}),
    "S02": ("schools", G + "schools-admissions", "by 31 October", {
        "en": "What is the last date to apply for a secondary school place?",
        "pl": "Do kiedy trzeba złożyć wniosek o miejsce w szkole średniej?",
        "ro": "Până la ce dată trebuie să aplic pentru un loc la școala secundară?",
        "ar": "ما هو آخر موعد للتقديم على مقعد في المدرسة الثانوية؟",
        "ur": "سیکنڈری اسکول میں داخلے کے لیے درخواست کس تاریخ تک دینی ہوگی؟"}),
    "S03": ("schools", G + "schools-admissions", "primary schools on 16 April", {
        "en": "When will I find out which primary school my child has been offered?",
        "pl": "Kiedy dowiem się, do której szkoły podstawowej zostało przyjęte moje dziecko?",
        "ro": "Când voi afla la ce școală primară a primit loc copilul meu?",
        "ar": "متى سأعرف المدرسة الابتدائية التي قُبل فيها طفلي؟",
        "ur": "مجھے کب پتہ چلے گا کہ میرے بچے کو کون سے پرائمری اسکول میں جگہ ملی ہے؟"}),
    "S04": ("schools", G + "schools-admissions", "at least 20 school days", {
        "en": "How long do I have to appeal if my child is refused a school place?",
        "pl": "Ile mam czasu na odwołanie, jeśli moje dziecko nie dostało miejsca w szkole?",
        "ro": "Cât timp am la dispoziție să fac contestație dacă copilul meu nu a primit loc la școală?",
        "ar": "كم لدي من الوقت للاستئناف إذا رُفض طلب طفلي للحصول على مقعد في المدرسة؟",
        "ur": "اگر میرے بچے کو اسکول میں جگہ نہ ملے تو اپیل کرنے کے لیے میرے پاس کتنا وقت ہے؟"}),
    "S05": ("schools", G + "school-attendance-absence", "fined or face legal action for taking your child on holiday", {
        "en": "Can I be fined for taking my child on holiday during term time?",
        "pl": "Czy mogę dostać grzywnę za zabranie dziecka na wakacje w trakcie roku szkolnego?",
        "ro": "Pot primi amendă dacă îmi iau copilul în vacanță în timpul semestrului?",
        "ar": "هل يمكن أن أُغرَّم إذا أخذت طفلي في عطلة أثناء الفصل الدراسي؟",
        "ur": "کیا ٹرم کے دوران بچے کو چھٹیوں پر لے جانے پر مجھے جرمانہ ہو سکتا ہے؟"}),
    "S06": ("schools", G + "apply-free-school-meals", "free school meals if you get any of the following", {
        "en": "Can my child get free school meals if I receive Universal Credit?",
        "pl": "Czy moje dziecko może dostać darmowe posiłki w szkole, jeśli otrzymuję Universal Credit?",
        "ro": "Poate copilul meu să primească mese gratuite la școală dacă primesc Universal Credit?",
        "ar": "هل يمكن لطفلي الحصول على وجبات مدرسية مجانية إذا كنت أتلقى Universal Credit؟",
        "ur": "اگر مجھے Universal Credit ملتا ہے تو کیا میرے بچے کو اسکول میں مفت کھانا مل سکتا ہے؟"}),
    "S07": ("schools", G + "free-school-transport", "more than 2 miles away and the child is under 8", {
        "en": "Is school transport free if the school is far from home?",
        "pl": "Czy transport do szkoły jest darmowy, jeśli szkoła jest daleko od domu?",
        "ro": "Este transportul la școală gratuit dacă școala este departe de casă?",
        "ar": "هل المواصلات المدرسية مجانية إذا كانت المدرسة بعيدة عن المنزل؟",
        "ur": "اگر اسکول گھر سے دور ہو تو کیا اسکول کی ٹرانسپورٹ مفت ہے؟"}),
    "S08": ("schools", G + "know-when-you-can-leave-school", "16 by the end of the summer holidays", {
        "en": "At what age can my child leave school?",
        "pl": "W jakim wieku moje dziecko może skończyć szkołę?",
        "ro": "La ce vârstă poate copilul meu să termine școala?",
        "ar": "في أي عمر يمكن لطفلي ترك المدرسة؟",
        "ur": "میرا بچہ کس عمر میں اسکول چھوڑ سکتا ہے؟"}),
    "S09": ("schools", G + "children-with-special-educational-needs", "SEN support - support given in school", {
        "en": "What support can my child get for special educational needs?",
        "pl": "Jakie wsparcie może otrzymać moje dziecko ze specjalnymi potrzebami edukacyjnymi?",
        "ro": "Ce sprijin poate primi copilul meu pentru cerințe educaționale speciale?",
        "ar": "ما الدعم الذي يمكن أن يحصل عليه طفلي إذا كانت لديه احتياجات تعليمية خاصة؟",
        "ur": "خصوصی تعلیمی ضروریات والے میرے بچے کو کیا مدد مل سکتی ہے؟"}),
    "S10": ("schools", G + "complain-about-school", "different ways to complain in England", {
        "en": "How do I complain about my child's state school?",
        "pl": "Jak złożyć skargę na państwową szkołę mojego dziecka?",
        "ro": "Cum fac o plângere împotriva școlii de stat a copilului meu?",
        "ar": "كيف أقدم شكوى ضد المدرسة الحكومية التي يدرس فيها طفلي؟",
        "ur": "میں اپنے بچے کے سرکاری اسکول کے بارے میں شکایت کیسے کروں؟"}),

    "H01": ("housing", G + "guidance/renters-rights-act-overview-for-tenants", "section 21 process", {
        "en": "Can my landlord evict me without giving a reason?",
        "pl": "Czy wynajmujący może mnie eksmitować bez podania powodu?",
        "ro": "Poate proprietarul să mă evacueze fără să-mi dea un motiv?",
        "ar": "هل يمكن للمالك أن يطردني دون إبداء سبب؟",
        "ur": "کیا مالک مکان مجھے بغیر کوئی وجہ بتائے گھر سے نکال سکتا ہے؟"}),
    "H02": ("housing", G + "guidance/renters-rights-act-overview-for-tenants", "increase the rent once a year", {
        "en": "How often can my landlord increase the rent?",
        "pl": "Jak często wynajmujący może podnosić czynsz?",
        "ro": "Cât de des poate proprietarul să crească chiria?",
        "ar": "كم مرة يمكن للمالك أن يرفع الإيجار؟",
        "ur": "مالک مکان کتنی بار کرایہ بڑھا سکتا ہے؟"}),
    "H03": ("housing", G + "tenancy-deposit-protection", "Deposit Protection Service", {
        "en": "My landlord took a deposit from me. Does it have to be protected?",
        "pl": "Wynajmujący pobrał ode mnie kaucję. Czy musi ona być zabezpieczona?",
        "ro": "Proprietarul mi-a luat o garanție. Trebuie aceasta să fie protejată?",
        "ar": "أخذ المالك مني مبلغ تأمين، هل يجب أن يكون هذا التأمين محميًا؟",
        "ur": "مالک مکان نے مجھ سے ڈپازٹ لیا ہے، کیا اس ڈپازٹ کا محفوظ ہونا ضروری ہے؟"}),
    "H04": ("housing", G + "housing-benefit", "help you pay your rent", {
        "en": "How can I get help paying my rent if I am on a low income?",
        "pl": "Jak uzyskać pomoc w opłaceniu czynszu, jeśli mam niskie dochody?",
        "ro": "Cum pot primi ajutor pentru plata chiriei dacă am un venit mic?",
        "ar": "كيف أحصل على مساعدة في دفع الإيجار إذا كان دخلي منخفضًا؟",
        "ur": "کم آمدنی کی صورت میں مجھے کرایہ ادا کرنے میں مدد کیسے مل سکتی ہے؟"}),
    "H05": ("housing", G + "council-housing", "join a waiting list", {
        "en": "How do I apply for council housing?",
        "pl": "Jak złożyć wniosek o mieszkanie komunalne?",
        "ro": "Cum aplic pentru o locuință socială de la consiliul local?",
        "ar": "كيف أتقدم بطلب للحصول على سكن من المجلس المحلي؟",
        "ur": "میں کونسل کے مکان کے لیے درخواست کیسے دوں؟"}),
    "H06": ("housing", G + "homelessness-help-from-council", "homeless or about to lose your home", {
        "en": "I am about to lose my home. Can the council help me?",
        "pl": "Zaraz stracę dom. Czy lokalny urząd może mi pomóc?",
        "ro": "Sunt pe punctul de a-mi pierde locuința. Mă poate ajuta consiliul local?",
        "ar": "أنا على وشك أن أفقد منزلي، هل يمكن للمجلس المحلي مساعدتي؟",
        "ur": "میرا گھر چھننے والا ہے، کیا کونسل میری مدد کر سکتی ہے؟"}),
    "H07": ("housing", G + "private-renting", "safe and in a good state of repair", {
        "en": "Does my landlord have to keep my rented home in good repair?",
        "pl": "Czy wynajmujący musi utrzymywać wynajmowane mieszkanie w dobrym stanie technicznym?",
        "ro": "Este proprietarul obligat să mențină locuința închiriată în stare bună?",
        "ar": "هل يجب على المالك أن يحافظ على المنزل المؤجر في حالة جيدة؟",
        "ur": "کیا مالک مکان پر کرائے کے گھر کو اچھی حالت میں رکھنا لازم ہے؟"}),
    "H08": ("housing", G + "private-renting", "24 hours", {
        "en": "How much notice must my landlord give before visiting the property?",
        "pl": "Z jakim wyprzedzeniem wynajmujący musi mnie uprzedzić przed wizytą w mieszkaniu?",
        "ro": "Cu cât timp înainte trebuie să mă anunțe proprietarul dacă vrea să vină în locuință?",
        "ar": "كم من الوقت يجب أن يخطرني المالك قبل أن يزور المنزل؟",
        "ur": "گھر آنے سے پہلے مالک مکان کو مجھے کتنا پہلے اطلاع دینی چاہیے؟"}),
    "H09": ("housing", G + "council-tax", "25% off your bill", {
        "en": "Do I get a council tax discount if I live alone?",
        "pl": "Czy dostanę zniżkę na podatek lokalny, jeśli mieszkam sam?",
        "ro": "Primesc reducere la impozitul local pe locuință dacă locuiesc singur?",
        "ar": "هل أحصل على خصم في ضريبة المجلس المحلي إذا كنت أعيش وحدي؟",
        "ur": "اگر میں اکیلا رہتا ہوں تو کیا مجھے کونسل ٹیکس میں رعایت ملتی ہے؟"}),
    "H10": ("housing", G + "council-housing-association-evictions", "stop or delay the eviction", {
        "en": "What can I do if my council or housing association wants to evict me?",
        "pl": "Co mogę zrobić, jeśli lokalny urząd lub stowarzyszenie mieszkaniowe chce mnie eksmitować?",
        "ro": "Ce pot face dacă consiliul local sau asociația de locuințe vrea să mă evacueze?",
        "ar": "ماذا يمكنني أن أفعل إذا أراد المجلس المحلي أو جمعية الإسكان طردي؟",
        "ur": "اگر کونسل یا ہاؤسنگ ایسوسی ایشن مجھے گھر سے نکالنا چاہے تو میں کیا کر سکتا ہوں؟"}),

    "M01": ("health", N + "gps/how-to-register-with-a-gp-surgery/", "change your current GP surgery online", {
        "en": "How do I register with a GP surgery?",
        "pl": "Jak zarejestrować się u lekarza rodzinnego?",
        "ro": "Cum mă înregistrez la un medic de familie?",
        "ar": "كيف أسجل لدى طبيب عام؟",
        "ur": "میں جی پی کے پاس رجسٹریشن کیسے کرواؤں؟"}),
    "M02": ("health", N + "gps/how-to-register-with-a-gp-surgery/", "do not need ID, proof of address", {
        "en": "Do I need proof of address or immigration status to register with a GP?",
        "pl": "Czy potrzebuję potwierdzenia adresu lub statusu imigracyjnego, aby zarejestrować się u lekarza rodzinnego?",
        "ro": "Am nevoie de dovada adresei sau a statutului de imigrant ca să mă înregistrez la medicul de familie?",
        "ar": "هل أحتاج إلى إثبات عنوان أو إثبات وضع الهجرة للتسجيل لدى طبيب عام؟",
        "ur": "کیا جی پی کے پاس رجسٹر ہونے کے لیے مجھے پتے یا امیگریشن اسٹیٹس کا ثبوت چاہیے؟"}),
    "M03": ("health", N + "urgent-and-emergency-care-services/when-to-use-111/", "if you think you need medical help right now", {
        "en": "When should I use NHS 111?",
        "pl": "Kiedy powinienem skorzystać z NHS 111?",
        "ro": "Când ar trebui să folosesc NHS 111?",
        "ar": "متى يجب أن أستخدم خدمة NHS 111؟",
        "ur": "مجھے NHS 111 کب استعمال کرنا چاہیے؟"}),
    "M04": ("health", N + "urgent-and-emergency-care-services/when-to-go-to-ae/", "serious injuries and life-threatening emergencies", {
        "en": "When should I go to A&E?",
        "pl": "Kiedy powinienem iść na szpitalny oddział ratunkowy?",
        "ro": "Când ar trebui să merg la urgențe?",
        "ar": "متى يجب أن أذهب إلى قسم الطوارئ؟",
        "ur": "مجھے ہسپتال کے ایمرجنسی شعبے میں کب جانا چاہیے؟"}),
    "M05": ("health", N + "urgent-and-emergency-care-services/when-to-call-999/", "999 is for life-threatening emergencies", {
        "en": "When should I call 999?",
        "pl": "Kiedy powinienem dzwonić pod numer 999?",
        "ro": "Când ar trebui să sun la 999?",
        "ar": "متى يجب أن أتصل بالرقم 999؟",
        "ur": "مجھے 999 پر کب کال کرنی چاہیے؟"}),
    "M06": ("health", N + "dentists/how-to-find-an-nhs-dentist/", "Ask for an NHS appointment", {
        "en": "How can I find an NHS dentist?",
        "pl": "Jak znaleźć dentystę NHS?",
        "ro": "Cum găsesc un dentist NHS?",
        "ar": "كيف أجد طبيب أسنان تابعًا لـ NHS؟",
        "ur": "میں NHS کا ڈینٹسٹ کیسے تلاش کروں؟"}),
    "M07": ("health", N + "dentists/how-much-nhs-dental-treatment-costs/", "£27.90", {
        "en": "How much does a basic NHS dental check-up cost?",
        "pl": "Ile kosztuje podstawowy przegląd u dentysty w NHS?",
        "ro": "Cât costă un control stomatologic de bază la NHS?",
        "ar": "كم يكلف فحص الأسنان الأساسي في NHS؟",
        "ur": "NHS میں دانتوں کے بنیادی معائنے پر کتنا خرچ آتا ہے؟"}),
    "M08": ("health", N + "pharmacies/how-pharmacies-can-help/", "This is called Pharmacy First", {
        "en": "Can a pharmacy give me prescription medicine without seeing a GP?",
        "pl": "Czy apteka może wydać mi lek na receptę bez wizyty u lekarza?",
        "ro": "Poate o farmacie să-mi dea medicamente cu rețetă fără să merg la medic?",
        "ar": "هل يمكن للصيدلية أن تعطيني دواءً يحتاج إلى وصفة طبية دون أن أرى الطبيب؟",
        "ur": "کیا فارمیسی ڈاکٹر کو دکھائے بغیر مجھے نسخے والی دوا دے سکتی ہے؟"}),
    "M09": ("health", N + "help-with-health-costs/nhs-low-income-scheme-lis/", "HC2 certificate", {
        "en": "Can I get help with health costs if I have a low income?",
        "pl": "Czy mogę dostać pomoc w pokryciu kosztów leczenia, jeśli mam niskie dochody?",
        "ro": "Pot primi ajutor pentru costurile medicale dacă am venituri mici?",
        "ar": "هل يمكنني الحصول على مساعدة في التكاليف الصحية إذا كان دخلي منخفضًا؟",
        "ur": "کیا کم آمدنی کی صورت میں مجھے علاج کے اخراجات میں مدد مل سکتی ہے؟"}),
    "M10": ("health", G + "healthcare-immigration-application", "£776 per year for students", {
        "en": "How much is the immigration health surcharge for students?",
        "pl": "Ile wynosi dopłata zdrowotna dla imigrantów w przypadku studentów?",
        "ro": "Cât costă taxa de sănătate pentru imigranți în cazul studenților?",
        "ar": "كم تبلغ رسوم الرعاية الصحية للمهاجرين بالنسبة للطلاب؟",
        "ur": "طلبہ کے لیے امیگریشن ہیلتھ سرچارج کتنا ہے؟"}),
}


def main():
    pages = {}
    for line in open(ROOT / "data/processed/pages.jsonl"):
        r = json.loads(line)
        pages[r["url"]] = " ".join(s["text"] for s in r["sections"]).lower()
    bad = [(q, url) for q, (_, url, ev, _) in Q.items() if url not in pages or ev.lower() not in pages[url]]
    if bad:
        sys.exit(f"evidence not found on gold page: {bad}")
    out = ROOT / "eval/queries.csv"
    with out.open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["qid", "domain", "lang", "question", "gold_url", "evidence", "translation_status"])
        for qid, (domain, url, ev, qs) in Q.items():
            for lang, text in qs.items():
                w.writerow([qid, domain, lang, text, url, ev, "source" if lang == "en" else "unverified"])
    print(f"{len(Q)} questions x {len(next(iter(Q.values()))[3])} languages -> {out.relative_to(ROOT)}; "
          f"all evidence phrases found verbatim on their gold page")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile eval/evaluate_retrieval.py
"""Week 4 retrieval experiment.

Compares four retrieval set-ups on the same 150 queries (30 questions x 5 languages):
  bm25            - lexical baseline over section chunks
  e5_naive150     - the original notebook set-up: e5-base on blind 150-word chunks
  e5_sections     - e5-base on heading-aware section chunks (src/chunk.py)
  minilm_sections - paraphrase-multilingual-MiniLM-L12-v2 on the same section chunks

Metrics (all at page or chunk level as named):
  hit@1, hit@3  - gold page appears in the top 1 / top 3 distinct pages
  mrr@10        - mean reciprocal rank of the gold page among the top 10 distinct pages
  ans@3         - at least one of the top 3 chunks contains the verbatim answer phrase
Outputs: results/retrieval_by_lang.csv, results/retrieval_per_query.csv
"""
import json
import pathlib
import sys
import time

import pandas as pd

ROOT = pathlib.Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from retrieval import BM25Retriever, DenseRetriever  # noqa: E402

E5 = "intfloat/multilingual-e5-base"
MINILM = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


def naive_chunks():
    """Reproduce the notebook: whole page text cut every 150 words, no headings."""
    rows = []
    for line in open(ROOT / "data/processed/pages.jsonl"):
        p = json.loads(line)
        words = " ".join(s["text"] for s in p["sections"]).split()
        for i in range(0, len(words), 150):
            body = " ".join(words[i:i + 150])
            rows.append({"url": p["url"], "body": body, "text": body})
    return pd.DataFrame(rows)


def score(retriever, chunks, queries, name):
    rows = []
    for q in queries.itertuples():
        hits = retriever.search(q.question, k=30)
        urls = list(dict.fromkeys(chunks.iloc[i].url for i, _ in hits))[:10]
        rank = urls.index(q.gold_url) + 1 if q.gold_url in urls else None
        top3 = [chunks.iloc[i].body.lower() for i, _ in hits[:3]]
        rows.append({
            "system": name, "qid": q.qid, "domain": q.domain, "lang": q.lang,
            "hit@1": rank == 1, "hit@3": rank is not None and rank <= 3,
            "mrr@10": 1 / rank if rank else 0.0,
            "ans@3": any(q.evidence.lower() in t for t in top3),
            "top1_url": urls[0], "top1_score": round(hits[0][1], 4),
        })
    return rows


def main():
    queries = pd.read_csv(ROOT / "eval/queries.csv")
    sections = pd.read_csv(ROOT / "data/processed/chunks.csv")
    naive = naive_chunks()
    print(f"{len(queries)} queries | section chunks {len(sections)} | naive chunks {len(naive)}")

    systems = [
        ("bm25", lambda: BM25Retriever(sections.text.tolist()), sections),
        ("e5_naive150", lambda: DenseRetriever(E5, naive.text.tolist()), naive),
        ("e5_sections", lambda: DenseRetriever(E5, sections.text.tolist()), sections),
        ("minilm_sections", lambda: DenseRetriever(MINILM, sections.text.tolist()), sections),
    ]
    rows = []
    for name, build, chunks in systems:
        t0 = time.time()
        r = build()
        rows += score(r, chunks, queries, name)
        print(f"  {name:16} done in {time.time() - t0:.0f}s")
        if name == "e5_sections":
            r.save(ROOT / "data/processed/e5_sections.faiss")

    per_q = pd.DataFrame(rows)
    out = ROOT / "results"
    out.mkdir(exist_ok=True)
    per_q.to_csv(out / "retrieval_per_query.csv", index=False)
    metrics = ["hit@1", "hit@3", "mrr@10", "ans@3"]
    by_lang = per_q.groupby(["system", "lang"])[metrics].mean().round(3)
    overall = per_q.groupby("system")[metrics].mean().round(3)
    overall["lang"] = "ALL"
    table = pd.concat([by_lang.reset_index(), overall.reset_index()])
    table.to_csv(out / "retrieval_by_lang.csv", index=False)

    print("\nOverall (150 queries)\n", overall[metrics].to_string())
    print("\nhit@3 by language\n", per_q.pivot_table(index="system", columns="lang", values="hit@3").round(2).to_string())
    print("\nans@3 by language\n", per_q.pivot_table(index="system", columns="lang", values="ans@3").round(2).to_string())
    print("\nans@3 by domain\n", per_q.pivot_table(index="system", columns="domain", values="ans@3").round(2).to_string())


if __name__ == "__main__":
    main()


## Build the corpus, chunks and test set

In [ ]:
!{sys.executable} src/ingest.py

In [ ]:
!{sys.executable} src/chunk.py

In [ ]:
!{sys.executable} eval/build_queries.py

## Run the retrieval experiment (4 set-ups x 150 queries)

In [ ]:
!{sys.executable} eval/evaluate_retrieval.py 2>&1 | grep -v -i -E 'warn|unauthenticated|tqdm'

In [ ]:
import sys, json, pathlib
import pandas as pd
ROOT = pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
pd.set_option('display.max_colwidth', 90)
pages = pd.DataFrame([json.loads(l) for l in open(ROOT / 'data/processed/pages.jsonl')])
chunks = pd.read_csv(ROOT / 'data/processed/chunks.csv')
pages.groupby(['source', 'domain']).agg(pages=('id', 'count'), words=('words', 'sum'))

Publisher dates are not a reliable sign of how current a page is. The eviction guide still says it was published in 2015, but its text already describes the Renters' Rights Act changes from 1 May 2026.

In [ ]:
ev = pages[pages.title.str.startswith('Private renting for tenants')].iloc[0]
text = ' '.join(s['text'] for s in ev.sections)
print('public_updated_at:', ev.public_updated_at, '| mentions of "1 May 2026":', text.count('1 May 2026'))

## 3. Multilingual index

In [ ]:
from retrieval import DenseRetriever
e5 = DenseRetriever('intfloat/multilingual-e5-base', chunks.text.tolist())
print('chunks indexed:', e5.index.ntotal)

def show(question, k=3):
    print('Q:', question)
    for i, s in e5.search(question, k):
        c = chunks.iloc[i]
        print(f'  {s:.3f}  {c.title} > {c.heading}\n         {c.url}')
    print()

In [ ]:
show('How do I apply for a primary school place for my child?')   # English
show('Jak zarejestrować się u lekarza rodzinnego?')                 # Polish: how do I register with a GP?
show('Poate proprietarul să mă evacueze fără să-mi dea un motiv?')  # Romanian: can my landlord evict me without a reason?
show('هل يمكن للمالك أن يطردني من المنزل؟')                         # Arabic: can my landlord evict me?
show('میرا گھر چھننے والا ہے، کیا کونسل میری مدد کر سکتی ہے؟')      # Urdu: I am about to lose my home, can the council help?

## 4. Retrieval experiment

Test set `eval/queries.csv`: 30 questions (10 per domain) in English, Polish, Romanian, Arabic and Urdu = 150 queries. Each question has a gold page and an **answer phrase copied verbatim** from that page.

* **hit@3**: gold page in the top 3 pages
* **ans@3**: a top-3 chunk contains the answer phrase (strict: other wording may also answer)

Four set-ups: BM25 keyword search, the original notebook set-up (e5 on blind 150-word chunks), e5 on section chunks, and MiniLM on section chunks. Run `python eval/evaluate_retrieval.py` to regenerate.

In [ ]:
res = pd.read_csv(ROOT / 'results/retrieval_by_lang.csv')
res[res.lang == 'ALL'].drop(columns='lang').set_index('system')

In [ ]:
per_q = pd.read_csv(ROOT / 'results/retrieval_per_query.csv')
per_q.pivot_table(index='system', columns='lang', values=['hit@3', 'ans@3']).round(2)

In [ ]:
import matplotlib.pyplot as plt
order = ['bm25', 'e5_naive150', 'e5_sections', 'minilm_sections']
langs = ['en', 'pl', 'ro', 'ar', 'ur']
t = per_q.pivot_table(index='lang', columns='system', values='ans@3').loc[langs, order]
ax = t.plot.bar(figsize=(8, 3.5), rot=0, width=0.8)
ax.set_ylabel('ans@3'); ax.set_ylim(0, 1); ax.set_title('Answer-bearing chunk in top 3, by query language')
ax.legend(frameon=False, ncol=4, fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.12))
plt.tight_layout(); plt.show()

## 5. Failure cases

**Wrong nation.** For *At what age can my child leave school?* the top chunk is the Scotland rule. The page is right, but the answer would be wrong for someone in England.

In [ ]:
show('At what age can my child leave school?')

**Right page, not the answering section.** Page-level hit@3 is 0.96 but ans@3 is 0.67. Some of the gap is the strict phrase match (the top chunks for *Can my landlord evict me without giving a reason?* do explain that a reason is now needed), and some is real: the chunk with the answer ranks below other sections of the same page.

In [ ]:
e = per_q[per_q.system == 'e5_sections']
print('hit@3 but no answer phrase in top 3:', ((e['hit@3']) & (~e['ans@3'])).sum(), 'of', len(e))
e.groupby('lang')[['hit@3', 'ans@3']].mean().round(2).loc[langs]

## Next steps (week 5)

* Tag chunks by UK nation and filter to England by default.
* Native-speaker check of the Polish, Romanian, Arabic and Urdu test questions.
* Compare direct multilingual retrieval with translating the question to English first (NLLB-200).
* Try reranking the top 20 chunks to close the gap between hit@3 and ans@3, especially for Urdu.